In [1]:
# pip install langchain_huggingface 
# !pip install langchain_community

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import spacy
import re
import contractions
from textblob import TextBlob
import os

C:\Users\vigne\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\vigne\AppData\Local\Temp\ipykernel_4752\2617388848.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 1. Load the document(.txt)

In [3]:
data = open('data.txt').read()

## 2. Text Normalization

### Converting all the characters into lower case

In [4]:
data = data.lower()

### Removing extra spaces

In [5]:
data = re.sub(r'\s{2,}','',data)

### Removing numbers like 1.,2.

In [6]:
data = re.sub(r'\b\d+\.\b','',data)
data

'machine learning (ml) and deep aðÿ˜‚ðÿ˜‚learning (dl) are both importantsubsets of artificial intelligence, but they serve different purposes and approaches. let\'s break down each concept with real-time examples:### machine learning (ml):**definition:**\nmachine learning is a type of ai that allows computers to automatically learn and improve from experience without being explicitly programmed. this means that algorithms can be trained to recognize patterns in data and make predictions or decisions.**key concepts:**\n- **supervised learning:** where a model is trained on labeled data to predict outcomes. example: image classification (distinguishing between different types of objects).\n- **unsupervised learning:** where the model finds patterns in data without being told what to look for. example: clustering (grouping similar items together).\n- **reinforcement learning:** where an agent learns by interacting with an environment, trying to maximize a reward. example: self-driving ca

### Contractions

In [7]:
data = contractions.fix(data)

### Removing punctuations & spl characters

In [8]:
data = re.sub(r'[^0-9a-zA-Z\s]','',data)

### TextBlob

In [9]:
# values= TextBlob(data).correct()
# values

### Lemmatization

In [10]:
import spacy

nlp = spacy.load('en_core_web_sm')
tokens = nlp(data)
updated_tokens = [token.lemma_ for token in tokens if not token.is_stop]
data = ' '.join(updated_tokens).strip()
data

'machine learn ml deep alearning dl importantsubset artificial intelligence serve different purpose approach let break concept realtime example machine learn mldefinition \n machine learning type ai allow computer automatically learn improve experience explicitly program mean algorithm train recognize pattern datum prediction decisionskey concept \n  supervise learning model train label datum predict outcome example image classification distinguish different type object \n  unsupervised learning model find pattern datum tell look example cluster group similar item \n  reinforcement learning agent learn interact environment try maximize reward example selfdrive car learn navigate decision base feedbackrealtime example \n imagine selfdrive car drive safely learn experience car use sensor like camera radar detect object obstacle realtime use machine learn algorithm predict car base information collect allow car informed decision slow stop change lane avoid accident deep learning dldefinit

### Chunking(Converting doc -> chunks)

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap = 40
)

chunks = splitter.create_documents([data])
chunks

[Document(metadata={}, page_content='machine learn ml deep alearning dl importantsubset artificial intelligence serve different purpose approach let break concept realtime example machine learn mldefinition'),
 Document(metadata={}, page_content='machine learning type ai allow computer automatically learn improve experience explicitly program mean algorithm train recognize pattern datum prediction decisionskey concept'),
 Document(metadata={}, page_content='supervise learning model train label datum predict outcome example image classification distinguish different type object'),
 Document(metadata={}, page_content='unsupervised learning model find pattern datum tell look example cluster group similar item'),
 Document(metadata={}, page_content='reinforcement learning agent learn interact environment try maximize reward example selfdrive car learn navigate decision base feedbackrealtime example'),
 Document(metadata={}, page_content='imagine selfdrive car drive safely learn experience 

In [12]:
chunks[0].metadata={'file_name':'data.txt'}
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='machine learn ml deep alearning dl importantsubset artificial intelligence serve different purpose approach let break concept realtime example machine learn mldefinition'),
 Document(metadata={}, page_content='machine learning type ai allow computer automatically learn improve experience explicitly program mean algorithm train recognize pattern datum prediction decisionskey concept'),
 Document(metadata={}, page_content='supervise learning model train label datum predict outcome example image classification distinguish different type object'),
 Document(metadata={}, page_content='unsupervised learning model find pattern datum tell look example cluster group similar item'),
 Document(metadata={}, page_content='reinforcement learning agent learn interact environment try maximize reward example selfdrive car learn navigate decision base feedbackrealtime example'),
 Document(metadata={}, page_content='imagine selfdrive car drive s

### Chunk Embeddings (Converts chunks to vectors)

In [13]:
chunks

[Document(metadata={'file_name': 'data.txt'}, page_content='machine learn ml deep alearning dl importantsubset artificial intelligence serve different purpose approach let break concept realtime example machine learn mldefinition'),
 Document(metadata={}, page_content='machine learning type ai allow computer automatically learn improve experience explicitly program mean algorithm train recognize pattern datum prediction decisionskey concept'),
 Document(metadata={}, page_content='supervise learning model train label datum predict outcome example image classification distinguish different type object'),
 Document(metadata={}, page_content='unsupervised learning model find pattern datum tell look example cluster group similar item'),
 Document(metadata={}, page_content='reinforcement learning agent learn interact environment try maximize reward example selfdrive car learn navigate decision base feedbackrealtime example'),
 Document(metadata={}, page_content='imagine selfdrive car drive s

In [15]:
# from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='google/embeddinggemma-300m'
)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 3178.13it/s]


In [16]:
vectordb = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
vectordb

In [17]:
user_query = 'What is Machine Learning ?'
r_chunks = vectordb.similarity_search(user_query)

In [18]:
updated_r_chunks = set()

for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)

R_text ='\n'.join(updated_r_chunks)
R_text

'eg time place construct appropriate responsein summary machine learning make machine well task human need lot practice deep learning make machine well recognize complex pattern feature datum\ndeep learning specific type machine learning use neural network layer deep dl layer enable complex feature extract datakey concept\nmachine learning type ai allow computer automatically learn improve experience explicitly program mean algorithm train recognize pattern datum prediction decisionskey concept\nmachine learn ml deep alearning dl importantsubset artificial intelligence serve different purpose approach let break concept realtime example machine learn mldefinition'

In [21]:
from langchain_ollama import ChatOllama
def r_search(query,k=3):
    R_chunks = vectordb.similarity_search(query,k=k)
    # print(R_chunks)
    R_chunks = {doc.page_content for doc in R_chunks}
    R_Text = '\n'.join(R_chunks)
    return R_Text
def g_text(r_search,query):
        import os
        prompt = f'''
                    You are a strict data structuring assistant. Your sole task is to structure the provided source text based on the user's input.

                    Source Text to structure:
                    {r_search}

                    User Input:
                    {query}

                    STRICT RULES FOR ZERO HALLUCINATION:
                    1. NO OUTSIDE KNOWLEDGE: You must use ONLY the information explicitly stated in the Source Text. Do not add any external facts, assumptions, or extra content.
                    2. DO NOT CORRECT THE DATA: If there is a grammatical, spelling, or factual mistake in the Source Text, you MUST preserve it exactly as written. Do not attempt to fix or alter the original meaning.
                    3. NO CONVERSATIONAL FILLER: Do not output introductory or concluding remarks (e.g., "Here is the output").
                    4. INSUFFICIENT CONTEXT: If the Source Text does not contain enough information to structure an answer for the input, output exactly: "Insufficient context."

                    Expected Output Format:
                    Input : {query}
                    Output : [Insert purely structured output here]
            '''
        # llm_model = ChatGoogleGenerativeAI(
        #     model="gemini-3.5-flash", #gemini-3.5-flash,gemini-3.6-flash,gemini-2.5-pro
        #     api_key=os.environ['GEMINI_API_KEY'],
        #     temperature = 0.0
        # )
            
        llm_model = ChatOllama(
            model="llama3.1",
            temperature=0.0
        )

        response = llm_model.invoke(prompt).content
        return response
user_prompt = 'Explain Machine Learning ?'
user_prompt = re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
r_response = r_search(user_prompt)
g_response = g_text(r_response,user_prompt)
print(g_response)

Input : Explain Machine Learning 
Output : 
Machine Learning: 
Type of AI that allows a computer to automatically learn and improve from experience without being explicitly programmed.
Algorithm trains to recognize patterns and make predictions.
Key concept: Machine Learning is a subset of Artificial Intelligence that enables computers to learn and improve from data.

Deep Learning: 
Specific type of Machine Learning that uses a neural network with multiple layers to enable complex feature extraction and data analysis.
Key concept: Deep Learning is a subset of Machine Learning that uses neural networks to analyze data.
